In [23]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Predict with the Flow Matching Model

In [ ]:
import importlib

import hydra
import lightning as L
import torch
import xarray as xr
from einops import rearrange, reduce
from omegaconf import DictConfig
from tqdm import tqdm

from genpp import BASE_DIR
from genpp.configs import register_resolvers
from genpp.data import OBSERVATIONS_FLAT_PATH
from genpp.eval.utils import log_scores
from genpp.models.loss import EnergyScore, EnsembleCRPS, VariogramScore

try:
    register_resolvers()
except Exception:
    pass

torch.set_float32_matmul_precision("medium")

In [5]:
# Model ID is generated by WandB
MODEL_ID = "kqf8bqi6"
OUTPUT_DIR = BASE_DIR.parent.parent / "outputs"

MODEL_DIR = list(OUTPUT_DIR.rglob(f"*{MODEL_ID}*"))[0].parent.parent.parent
MODEL_CHECKPOINT = list(MODEL_DIR.rglob("*.ckpt"))[0]

In [6]:
with hydra.initialize_config_dir(config_dir=str(MODEL_DIR / ".hydra"), version_base=None):
    cfg: DictConfig = hydra.compose(
        config_name="config",
    )
    # Do not shuffle any dataloader
    cfg.data.module.dataloader_config.train.shuffle = False
    cfg.data.module.dataloader_config.val.shuffle = False
    cfg.data.module.dataloader_config.test.shuffle = False
    datamodule = hydra.utils.instantiate(cfg.data.module)
datamodule.prepare_data()
datamodule.setup(stage="validate")

In [7]:
class_path = cfg.model._target_

module_path, class_name = class_path.rsplit(".", 1)
module = importlib.import_module(module_path)
ModelClass = getattr(module, class_name)

model = ModelClass.load_from_checkpoint(MODEL_CHECKPOINT)

/home/feik/GenPP/.pixi/envs/dev/lib/python3.13/site-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'loss_fn' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss_fn'])`.
/home/feik/GenPP/src/genpp/models/fm/fm_cnn.py:392: UserWarning: Ignoring additional arguments: {'loss_fn': EnergyScore(), 'rescaler': None}
  warn(f"Ignoring additional arguments: {kwargs}")


In [8]:
trainer = L.Trainer(logger=False, devices=[1])
pred_list = trainer.predict(model, datamodule.val_dataloader(), return_predictions=True)
predictions = torch.cat(pred_list, dim=0)  # type: ignore

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting DataLoader 0: 100%|██████████| 6/6 [02:44<00:00,  0.04it/s]


In [9]:
# Rescale the predictions (TODO this should happen in the predict step)
reverse_transform = datamodule.y_reverseModules[0]
mean = rearrange(reverse_transform.mean, "f -> 1 1 f 1 1")
scale = rearrange(reverse_transform.scale, "f -> 1 1 f 1 1")

predictions_rescaled = predictions * scale + mean

In [71]:
# load observations
val_split = hydra.utils.instantiate(cfg.data.splits.val)
y_val = xr.open_dataarray(OBSERVATIONS_FLAT_PATH).sel(time=val_split)
y_t = torch.from_numpy(y_val.values).to(predictions_rescaled)

In [72]:
# Compute the scores
crps_ens = EnsembleCRPS()
es = EnergyScore(clamp=False)
vs = VariogramScore(p=0.5)

crps_per_margin = crps_ens(predictions_rescaled, y_t)

# Rearrange so that we compute the scores seperatly for each variable
x_spatial = rearrange(predictions_rescaled, "t n d lat lon -> t d n (lat lon)")
y_spatial = rearrange(y_t, "t d lat lon -> t d (lat lon)")

energy_score_per_var = es(x_spatial, y_spatial)

# For the VS there are huge intermediary results, thats why it is computed batchwise
vss = []
for x_i, y_i in tqdm(zip(x_spatial, y_spatial), total=predictions_rescaled.shape[0]):
    vss.append(vs(x_i, y_i))
variogram_score_per_var = torch.stack(vss)


# Rearrange to compute scores for both variables together
x_full = rearrange(predictions_rescaled, "t n d lat lon -> t n (d lat lon)")
y_full = rearrange(y_t, "t d lat lon -> t (d lat lon)")

energy_score_full = es(x_full, y_full)

vss = []
for x_i, y_i in tqdm(zip(x_full, y_full), total=predictions_rescaled.shape[0]):
    vss.append(vs(x_i, y_i))
variogram_score_full = torch.stack(vss)

100%|██████████| 730/730 [02:06<00:00,  5.79it/s]


In [75]:
# Reduce the scores
variogram_score_per_var = reduce(variogram_score_per_var, "t d -> d", "mean")
energy_score_per_var = reduce(energy_score_per_var, "t d -> d", "mean")
crps_per_var = reduce(crps_per_margin, "t d h w -> d", reduction="mean")

variogram_score_full = reduce(variogram_score_full, "t -> 1", "mean")
energy_score_full = reduce(energy_score_full, "t -> 1", "mean")
crps_full = reduce(crps_per_margin, "t d h w -> 1", "mean")

In [ ]:
# Log the Scores
model_class = cfg.model._target_.split(".")[-1]
file = MODEL_DIR / "scores.csv"
# Log scores per Variable
log_scores(
    file=file,
    model=model_class,
    metric="CRPS",
    variables=datamodule.y_select_variables,
    scores=crps_per_var,
)
log_scores(
    file=file,
    model=model_class,
    metric="EnergyScore",
    variables=datamodule.y_select_variables,
    scores=energy_score_per_var,
)
log_scores(
    file=file,
    model=model_class,
    metric="VariogramScore",
    variables=datamodule.y_select_variables,
    scores=variogram_score_per_var,
)
# Log full scores
log_scores(file=file, model=model_class, metric="CRPS", variables=["combined"], scores=crps_full)
log_scores(
    file=file,
    model=model_class,
    metric="EnergyScore",
    variables=["combined"],
    scores=energy_score_full,
)
log_scores(
    file=file,
    model=model_class,
    metric="VariogramScore",
    variables=["combined"],
    scores=variogram_score_full,
)